# Vorflow Comprehensive Demo

This notebook walks through the main features of `vorflow` for generating unstructured Voronoi grids for MODFLOW 6.

## Workflow
1. **Define Geometry**: Create Shapely objects (Polygons, Lines, Points).
2. **ConceptualMesh**: Register these objects with refinement parameters.
3. **MeshGenerator**: Generate a triangular mesh using Gmsh.
4. **VoronoiTessellator**: Convert the triangles to a Voronoi grid.
5. **Analysis**: Check barriers, mesh quality and boundary-cell centering.

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Point, Polygon, box
from shapely.ops import unary_union

from vorflow import ConceptualMesh, GeometricGrowthField, MeshGenerator, VoronoiTessellator
from vorflow.utils import (
    boundary_connectivity_report,
    build_connectivity,
    calculate_mesh_quality,
    summarize_quality,
)

## 1. Define Geometry
The domain has a hole, a river, a fault, two wells, a refined zone, and a
refinement area that only sizes the mesh (it is not a zone).

In [ ]:
# Domain: 200x200 box with a square hole
hole = box(80, 80, 120, 120)
domain = Polygon(box(0, 0, 200, 200).exterior.coords, [hole.exterior.coords])

# Refined zone: a smaller box inside
zone_poly = box(50, 50, 75, 75)

# Refinement-only area, e.g. around planned pumping
refine_area = box(140, 20, 185, 60)

# River: a sine wave that starts and ends outside the domain
river_line = LineString([(x, 75 + 20 * np.sin(0.1 * x)) for x in range(-10, 220, 10)])

# Fault: a straight line through the hole (barrier)
fault_line = LineString([(100, 0), (100, 150)])

# Wells
well_point1 = Point(25, 25)
well_point2 = Point(60, 60)

## 2. Build Conceptual Mesh
Mesh size is set per feature:
- **resolution**: target cell size on the feature.
- **growth_factor**: how fast cells grow away from the feature (default 1.2), until they reach `background_lc`.
- **fields**: an explicit list of `MeshField` objects that replaces the default growth field.
- **embed=False**: the feature sizes the mesh but is not meshed geometry. A field-only polygon never becomes a zone and does not cut the domain.
- **z_order**: where polygons overlap, the higher value wins.
- **is_barrier=True**: cell faces line up along the line, so no face crosses it.

`dist_min`/`dist_max` are deprecated; they still work but select the old linear transition and emit a `DeprecationWarning`.

In [ ]:
cm = ConceptualMesh(crs="EPSG:3857")

# 1. Domain (background)
cm.add_polygon(domain, zone_id=1, resolution=20.0)

# 2. Refined zone; z_order=2 cuts it out of the domain polygon
cm.add_polygon(zone_poly, zone_id=2, z_order=2, resolution=3.0)

# 3. Refinement-only area: finer cells, but still zone 1
cm.add_polygon(refine_area, zone_id=99, resolution=6.0, embed=False)

# 4. River with a gentler explicit growth field
cm.add_line(river_line, line_id="river", resolution=2.0,
            fields=[GeometricGrowthField(growth_factor=1.1)])

# 5. Fault (barrier)
cm.add_line(fault_line, line_id="fault", resolution=2.0, is_barrier=True)

# 6. Wells; a larger growth_factor keeps the refinement local
cm.add_point(well_point1, point_id="well-a", resolution=1.0)
cm.add_point(well_point2, point_id="well-b", resolution=1.0, growth_factor=1.4)

# Process inputs
clean_polys, clean_lines, clean_pts = cm.generate()
clean_lines[["line_id", "lc", "is_barrier"]]

The fault crosses the hole, so clipping splits it into two lines. Both keep the `fault` ID.

## 3. Generate Triangular Mesh
Gmsh builds the triangulation. `verbosity=2` on `MeshGenerator` prints `[DIAG]` details while `generate()` runs.

In [ ]:
mg = MeshGenerator(background_lc=20.0)
mg.generate(clean_polys, clean_lines, clean_pts)
print(f"{len(mg.nodes)} mesh nodes")

## 4. Voronoi Tessellation
Convert the triangular mesh to a Voronoi grid, clipped to the domain.

In [ ]:
vt = VoronoiTessellator(mg, cm, clip_to_boundary=True)
grid = vt.generate()
print(f"{len(grid)} cells")
print("zones:", grid["zone_id"].value_counts().to_dict())

Zone 99 does not appear: the field-only polygon only refines the mesh.

## 5. Visualization
Plot the resulting grid along with the input features.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_aspect('equal')

# Plot grid
grid.plot(ax=ax, edgecolor='black', linewidth=0.5, alpha=0.5, color='white')

# Plot features
gpd.GeoSeries([domain]).boundary.plot(ax=ax, color='black', linewidth=2, label='Domain')
gpd.GeoSeries([zone_poly]).boundary.plot(ax=ax, color='blue', linestyle='--', label='Refined zone')
gpd.GeoSeries([refine_area]).boundary.plot(ax=ax, color='green', linestyle=':', label='Refinement only (embed=False)')
gpd.GeoSeries([river_line]).plot(ax=ax, color='cyan', linewidth=2, label='River')
gpd.GeoSeries([fault_line]).plot(ax=ax, color='red', linewidth=2, linestyle='-', label='Fault (barrier)')
gpd.GeoSeries([well_point1, well_point2]).plot(ax=ax, color='red', marker='*', markersize=100, label='Wells')

ax.set_xlim(-5, 205)
ax.set_ylim(-5, 205)
plt.legend()
plt.title("Voronoi Grid with Refinement")
plt.show()

## 6. Barriers
Barriers are enforced in two steps:
1. **Virtual straddle (mesh step).** The fault is not meshed as a line. Instead, pairs
   of nodes are placed just either side of it, so the Voronoi face between each
   pair lies on the fault.
2. **Cell splitting (tessellation step).** Any cell the fault still crosses is split.
   Each piece off the primary side gets a *mirror generator*, the primary generator
   reflected across the fault, so every face is still a perpendicular bisector.
   `vt.n_barrier_mirrors` counts these cells.

The connectivity report has one row per shared face (`shared_edge`) with the
generator-to-generator `connector`. With the barrier enforced, no face crosses the fault.
Faces that lie *on* the fault are where a horizontal flow barrier (MODFLOW 6 HFB
package) goes; their connectors cross the fault.

In [ ]:
connectivity = build_connectivity(grid, center="generator")
connectors = gpd.GeoSeries(connectivity["connector"], crs=grid.crs)
faces_crossing = connectivity.geometry.crosses(fault_line)
faces_on_fault = connectivity[connectivity.geometry.within(fault_line.buffer(1e-6))]
print(f"barrier mirror cells: {vt.n_barrier_mirrors}")
print(f"faces crossing the fault: {faces_crossing.sum()}")
print(f"faces on the fault (HFB candidates): {len(faces_on_fault)} of {len(connectivity)}")

fig, ax = plt.subplots(figsize=(8, 8))
grid.plot(ax=ax, edgecolor='black', linewidth=0.5, color='white')
ax.scatter(grid.x, grid.y, s=6, color='black', zorder=3, label='Generators')
connectors.plot(ax=ax, color='gray', linewidth=0.5, label='Connections')
gpd.GeoSeries([fault_line]).plot(ax=ax, color='red', linewidth=1, linestyle='--')
faces_on_fault.plot(ax=ax, color='red', linewidth=3, label='Faces on the fault')
ax.set_xlim(85, 115)
ax.set_ylim(20, 50)
ax.set_aspect('equal')
ax.legend()
ax.set_title("Cells and connections around the fault")
plt.show()

## 7. Quality Analysis
Check the geometric quality of the grid cells:
- `ortho_error`: angle between a connection and the normal of its face, in degrees (0 is ideal).
- `skewness`: CVFD face skewness, the offset of the connection-face intersection from the face midpoint over the face length (0 is ideal).
- `drift_ratio`: generator-to-centroid distance over √area (0 is ideal).

Passing the connectivity report from the previous step avoids rebuilding it.
Skewness is a per-face ratio, so very short faces (common where four generators are
nearly cocircular, and next to the barrier) can show large values while carrying
little flow.

In [ ]:
quality_gdf = calculate_mesh_quality(grid, calc_ortho=True, calc_skewness=True, connectivity=connectivity)
summarize_quality(quality_gdf)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (column, vmax, title) in zip(axes, [
    ("ortho_error", 10, "Orthogonality error (degrees)"),
    ("skewness", 0.3, "Face skewness"),
    ("drift_ratio", 0.4, "Drift ratio"),
]):
    quality_gdf.plot(column=column, ax=ax, legend=True, cmap='Reds', vmin=0, vmax=vmax)
    ax.set_aspect('equal')
    ax.set_title(title)
plt.show()

## 8. Boundary Cell Centering
Clipping a boundary cell to the domain leaves its generator on the clipped face, off the
cell's centroid. MODFLOW 6 connects cell centres, so the connections at the boundary
are less orthogonal when measured between centroids.

`boundary_centering="inset_mirror"` moves boundary generators inward and mirrors
ghost generators outside the domain, so the generator moves off the clipped face. The
original node positions are kept in `source_x`/`source_y`. The same mesh is reused
here; only the tessellation changes.

The report below compares centroid-to-centroid orthogonality for connections that
touch the boundary.

In [ ]:
vt_inset = VoronoiTessellator(mg, cm, clip_to_boundary=True, boundary_centering="inset_mirror")
grid_inset = vt_inset.generate()

domain_geom = unary_union(clean_polys[clean_polys["embed"].astype(bool)].geometry)
for label, g in [("clip", grid), ("inset_mirror", grid_inset)]:
    report = boundary_connectivity_report(g, domain_geom, center="centroid")
    print(f"{label:>12}: {len(report)} boundary connections, "
          f"ortho_error median {report['ortho_error'].median():.2f} deg, "
          f"95th percentile {report['ortho_error'].quantile(0.95):.2f} deg")

On this mesh `"inset_mirror"` lowers the median boundary orthogonality error from about
6.9 to 1.3 degrees (95th percentile 12.8 to 6.1). Nodes at sharp corners are not moved,
so the cells next to the domain and hole corners keep their clip-mode connections.
`boundary_inset_fraction` (default 0.25) sets the inward shift as a fraction of the local
boundary-node spacing; values near 0.5 overshoot and make Gmsh meshes worse than `"clip"`.

## Where next
- `cleaning_limitations_demo.ipynb`: common geometry problems (duplicate vertices, snapping, vertex spacing, slivers) and how to handle them.
- `field_capabilities_example.py`: every mesh-size field type, including `ThresholdField` and `ExponentialField`.
- `structured_buffer_example.py`: structured quad buffers along lines and polygons (`quad_buffer=True`).
- `triangular_grid_example.py`: exporting the triangular or mixed tri/quad mesh itself with `get_element_grid()`.
- `active_domain_quality_example.py`: meshing a larger envelope and selecting active cells afterwards.